# AI 3D + Animation Engine — One Colab

Two independent engines in one runtime:

- **3D Engine:** image → TRELLIS.2 → Unreal-safe PBR GLB + manifest
- **Animation Engine:** selected humanoid → MIA rig → ARDY → retarget → validated Unreal ZIP

Run either section by itself. If you run both, the Animation Engine can reuse the 3D Engine's GLB directly.


In [ ]:
# Shared setup — run once before either engine.
import pathlib, shutil, subprocess

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    text=True, capture_output=True, check=True
).stdout.strip().splitlines()
if not smi:
    raise RuntimeError("No NVIDIA GPU detected.")
gpu_name, memory_mib = [x.strip() for x in smi[0].rsplit(",", 1)]
memory_mib = int(memory_mib)
free_gib = shutil.disk_usage("/content").free / 1024**3
print(f"GPU: {gpu_name} | VRAM: {memory_mib/1024:.1f} GiB | Free disk: {free_gib:.1f} GiB")
if memory_mib < 24000:
    raise RuntimeError("Use a >=24 GB GPU for the supported pipeline.")
if free_gib < 35:
    raise RuntimeError("Need at least 35 GiB free disk for one engine.")

REPO = pathlib.Path("/content/My-works")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(
    ["git", "clone", "-q", "--depth", "1", "https://github.com/Logan17de/My-works.git", str(REPO)],
    check=True,
)
ENGINE_ROOT = REPO / "ai-3d-animation-engines"
TOOLS_3D = ENGINE_ROOT / "3d-engine"
TOOLS_ANIM = ENGINE_ROOT / "animation-engine"
print("Helpers ready.")


---
# 🎨 3D Engine

Run these cells when you want to create a 3D object/character.  
Objects can stop here; they are never sent to animation automatically.


In [ ]:
# Install TRELLIS.2 only.
subprocess.run(["bash", str(TOOLS_3D / "install_3d.sh")], check=True)


In [ ]:
# Upload image + set asset parameters.
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one reference image.")
INPUT_IMAGE = f"/content/{next(iter(uploaded))}"

ASSET_NAME = "asset" #@param {type:"string"}
ASSET_TYPE = "object" #@param ["object", "character", "environment", "other"]
TARGET_AXIS = "longest" #@param ["width", "height", "depth", "longest"]
TARGET_SIZE_METERS = 1.0 #@param {type:"number"}
DECIMATION_TARGET = 1000000 #@param {type:"integer"}
TEXTURE_SIZE = 4096 #@param {type:"integer"}

if TARGET_SIZE_METERS <= 0:
    raise ValueError("TARGET_SIZE_METERS must be positive.")


In [ ]:
# Generate Unreal-safe GLB.
import json, shlex

OUTPUT_3D_DIR = "/content/trellis_outputs"
cmd = [
    "/opt/conda/bin/conda", "run", "-n", "trellis2",
    "python", str(TOOLS_3D / "run_trellis2.py"),
    "--input", INPUT_IMAGE,
    "--output-dir", OUTPUT_3D_DIR,
    "--name", ASSET_NAME,
    "--asset-type", ASSET_TYPE,
    "--target-axis", TARGET_AXIS,
    "--target-size-m", str(TARGET_SIZE_METERS),
    "--envmap", "/content/TRELLIS.2/assets/hdri/forest.exr",
    "--decimation-target", str(DECIMATION_TARGET),
    "--texture-size", str(TEXTURE_SIZE),
]
print("Running:", " ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, cwd="/content/TRELLIS.2", check=True)

GLB_PATH = f"{OUTPUT_3D_DIR}/{ASSET_NAME}.glb"
ASSET_MANIFEST_PATH = f"{OUTPUT_3D_DIR}/{ASSET_NAME}_manifest.json"
PREVIEW_3D_PATH = f"{OUTPUT_3D_DIR}/{ASSET_NAME}_preview.mp4"

for p in (GLB_PATH, ASSET_MANIFEST_PATH):
    if not pathlib.Path(p).is_file():
        raise RuntimeError(f"Missing output: {p}")

print(json.dumps(json.loads(pathlib.Path(ASSET_MANIFEST_PATH).read_text())["geometry"], indent=2))


In [ ]:
# Preview / optionally download the 3D result.
from IPython.display import Video, display

if pathlib.Path(PREVIEW_3D_PATH).is_file():
    display(Video(PREVIEW_3D_PATH, embed=True))

DOWNLOAD_3D_NOW = False #@param {type:"boolean"}
if DOWNLOAD_3D_NOW:
    files.download(GLB_PATH)
    files.download(ASSET_MANIFEST_PATH)


---
# 🕺 Animation Engine

You can start here without running the 3D Engine.

Choose either:
- reuse the 3D Engine GLB currently in this runtime, or
- upload another humanoid manually.


In [ ]:
# Install ARDY + Make-It-Animatable only.
subprocess.run(["bash", str(TOOLS_ANIM / "install_animation.sh")], check=True)


In [ ]:
# Select character and motion settings.
import getpass, os
from google.colab import files

USE_3D_ENGINE_OUTPUT = False #@param {type:"boolean"}

if USE_3D_ENGINE_OUTPUT:
    TARGET_CHARACTER = globals().get("GLB_PATH")
    if not TARGET_CHARACTER or not pathlib.Path(TARGET_CHARACTER).is_file():
        raise RuntimeError("No live 3D Engine output. Run the 3D section first or disable this option.")
    SOURCE_MANIFEST = globals().get("ASSET_MANIFEST_PATH")
else:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one humanoid GLB/FBX/OBJ/PLY.")
    TARGET_CHARACTER = f"/content/{next(iter(uploaded))}"
    SOURCE_MANIFEST = None

if pathlib.Path(TARGET_CHARACTER).suffix.lower() not in {".glb", ".fbx", ".obj", ".ply"}:
    raise ValueError("Unsupported humanoid format.")

HF_TOKEN = getpass.getpass("Hugging Face token (Llama 3 access): " ).strip()
if not HF_TOKEN:
    raise ValueError("HF token required.")
os.environ["HF_TOKEN"] = HF_TOKEN

PROMPT = "A person walks forward, stops, and waves with the right hand." #@param {type:"string"}
DURATION_SECONDS = 6.0 #@param {type:"number"}
SEED = 0 #@param {type:"integer"}
TARGET_ALREADY_RIGGED = False #@param {type:"boolean"}
MIA_NO_FINGERS = True #@param {type:"boolean"}


In [ ]:
# Run the complete Animation Engine.
import shlex
OUTPUT_ANIM_DIR = "/content/animation_outputs"

cmd = [
    "python", str(TOOLS_ANIM / "run_animation_pipeline.py"),
    "--character", TARGET_CHARACTER,
    "--prompt", PROMPT,
    "--duration", str(DURATION_SECONDS),
    "--seed", str(SEED),
    "--output-dir", OUTPUT_ANIM_DIR,
]
if SOURCE_MANIFEST and pathlib.Path(SOURCE_MANIFEST).is_file():
    cmd += ["--source-manifest", SOURCE_MANIFEST]
if TARGET_ALREADY_RIGGED:
    cmd.append("--already-rigged")
if MIA_NO_FINGERS:
    cmd.append("--no-fingers")

print("Running:", " ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, env=os.environ.copy(), check=True)

MOTION_PREVIEW = f"{OUTPUT_ANIM_DIR}/motion_preview.mp4"
FINAL_FBX = f"{OUTPUT_ANIM_DIR}/character_animated.fbx"
CONTRACT_REPORT = f"{OUTPUT_ANIM_DIR}/animation_contract_report.json"
PACKAGE_ZIP = f"{OUTPUT_ANIM_DIR}/unreal_character_package.zip"

for p in (MOTION_PREVIEW, FINAL_FBX, CONTRACT_REPORT, PACKAGE_ZIP):
    if not pathlib.Path(p).is_file():
        raise RuntimeError(f"Missing animation output: {p}")

print("Animation Engine complete:", FINAL_FBX)


In [ ]:
# Preview motion and download the Unreal package.
from IPython.display import Video, display
from google.colab import files
display(Video(MOTION_PREVIEW, embed=True))
files.download(PACKAGE_ZIP)


---
## How to use this notebook

The heavy environments remain isolated:

```text
trellis2 → 3D Engine
ardy     → motion generation
mia      → rigging + Blender + retarget
```

So in the same Colab you can run:

- **3D only**
- **Animation only**
- **3D → manually choose its character → Animation**

No automatic object-to-animation flow is created.
